# 03 — sparkmeasure

https://github.com/LucaCanali/sparkMeasure

Dedicated notebook for measuring Spark jobs with `sparkMeasure`: stage metrics, task metrics, shuffle, joins, caching, skew, AQE, and benchmark comparison.

## 00 — Spark setup

Standard SparkSession setup used across the local Spark cluster notebooks.

In [1]:
from pyspark.sql import SparkSession, functions as F
import time
import pandas as pd

spark = (
    SparkSession.builder
    .appName("sparkmeasure")
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.debug.maxToStringFields", "1000")

spark

## 01 — Check sparkMeasure availability

This verifies that the sparkMeasure JVM classes are visible to the running Spark application.

In [2]:
def has_sparkmeasure():
    try:
        spark._jvm.ch.cern.sparkmeasure.StageMetrics
        spark._jvm.ch.cern.sparkmeasure.TaskMetrics
        return True
    except Exception:
        return False

SPARKMEASURE_AVAILABLE = has_sparkmeasure()
SPARKMEASURE_AVAILABLE

True

## 02 — Create metric helpers

Wrappers for StageMetrics, TaskMetrics, and simple fallback timing.

In [3]:
def timed(label, action):
    start = time.time()
    result = action()
    seconds = round(time.time() - start, 3)
    return {"case": label, "seconds": seconds, "result": result}

def stage_metrics(label, action):
    if not SPARKMEASURE_AVAILABLE:
        print(f"{label}: sparkMeasure is not available")
        return timed(label, action)

    sm = spark._jvm.ch.cern.sparkmeasure.StageMetrics(spark._jsparkSession)
    sm.begin()
    start = time.time()
    result = action()
    seconds = round(time.time() - start, 3)
    sm.end()
    print(f"\n--- {label} ---")
    sm.printReport()
    return {"case": label, "seconds": seconds, "result": result}

def task_metrics(label, action):
    if not SPARKMEASURE_AVAILABLE:
        print(f"{label}: sparkMeasure is not available")
        return timed(label, action)

    tm = spark._jvm.ch.cern.sparkmeasure.TaskMetrics(spark._jsparkSession)
    tm.begin()
    start = time.time()
    result = action()
    seconds = round(time.time() - start, 3)
    tm.end()
    print(f"\n--- {label} ---")
    tm.printReport()
    return {"case": label, "seconds": seconds, "result": result}

benchmark_rows = []

def record(row):
    benchmark_rows.append(row)
    return row

def results():
    return pd.DataFrame(benchmark_rows).drop(columns=["result"], errors="ignore").sort_values("seconds")

## 03 — Sample data

Synthetic fact/dimension datasets for repeatable measurements.

In [4]:
FACT_ROWS = 2_000_000
CUSTOMER_ROWS = 200_000
PRODUCT_ROWS = 50_000

fact_sales = (
    spark.range(0, FACT_ROWS)
    .withColumnRenamed("id", "sale_id")
    .withColumn("customer_id", (F.col("sale_id") % CUSTOMER_ROWS).cast("long"))
    .withColumn("product_id", (F.col("sale_id") % PRODUCT_ROWS).cast("long"))
    .withColumn("store_id", (F.col("sale_id") % 1_000).cast("long"))
    .withColumn("quantity", ((F.col("sale_id") % 5) + 1).cast("int"))
    .withColumn("amount", ((F.col("sale_id") % 500) + 10).cast("double"))
)

customers = (
    spark.range(0, CUSTOMER_ROWS)
    .withColumnRenamed("id", "customer_id")
    .withColumn("country", F.expr("CASE WHEN customer_id % 5 = 0 THEN 'SK' WHEN customer_id % 5 = 1 THEN 'CZ' WHEN customer_id % 5 = 2 THEN 'DE' WHEN customer_id % 5 = 3 THEN 'AT' ELSE 'PL' END"))
    .withColumn("segment", F.expr("CASE WHEN customer_id % 4 = 0 THEN 'enterprise' WHEN customer_id % 4 = 1 THEN 'smb' WHEN customer_id % 4 = 2 THEN 'consumer' ELSE 'public' END"))
)

products = (
    spark.range(0, PRODUCT_ROWS)
    .withColumnRenamed("id", "product_id")
    .withColumn("category", F.expr("CASE WHEN product_id % 4 = 0 THEN 'hardware' WHEN product_id % 4 = 1 THEN 'software' WHEN product_id % 4 = 2 THEN 'service' ELSE 'training' END"))
)

fact_sales.count(), customers.count(), products.count()

(2000000, 200000, 50000)

## 04 — Example 01: count baseline

Measures a basic distributed count.

In [6]:
record(stage_metrics("01_count_baseline", lambda: fact_sales.count()))
results()


--- 01_count_baseline ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 2
numTasks => 3
elapsedTime => 78 (78 ms)
stageDuration => 59 (59 ms)
executorRunTime => 12 (12 ms)
executorCpuTime => 9 (9 ms)
executorDeserializeTime => 32 (32 ms)
executorDeserializeCpuTime => 12 (12 ms)
resultSerializationTime => 0 (0 ms)
jvmGCTime => 0 (0 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
resultSize => 3850 (3.8 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 67174368
recordsRead => 2000000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2
shuffleTotalBlocksFetched => 2
shuffleLocalBlocksFetched => 2
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 118 (118 Bytes)
shuffleLocalBytesRead => 118 (118 Bytes)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWritten => 1

,case,seconds
1,01_count_baseline,0.115
0,01_count_baseline,0.138


## 05 — Example 02: filter pushdown-style workload

Measures a selective filter followed by count.

In [7]:
filtered_sales = fact_sales.filter(F.col("amount") > 300)

filtered_sales.explain("formatted")
record(stage_metrics("02_filter_count", lambda: filtered_sales.count()))
results()

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : (cast(((id#0L % 500) + 10) as double) > 300.0)

(3) Project [codegen id : 1]
Output [6]: [id#0L AS sale_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 50000) AS product_id#3L, (id#0L % 1000) AS store_id#4L, cast(((id#0L % 5) + 1) as int) AS quantity#5, cast(((id#0L % 500) + 10) as double) AS amount#6]
Input [1]: [id#0L]



--- 02_filter_count ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 2
numTasks => 3
elapsedTime => 113 (0.1 s)
stageDuration => 95 (95 ms)
executorRunTime => 87 (87 ms)
executorCpuTime => 64 (64 ms)
executorDeserializeTime => 33 (33 ms)
executorDeserializeCpuTime => 11 (11 ms)
resultSerializationTime => 0 (0 ms)
jvmGCTime => 0 (0 ms)
shuffleFetchWaitTime => 0 (0 

,case,seconds
1,01_count_baseline,0.115
0,01_count_baseline,0.138
2,02_filter_count,0.181


## 06 — Example 03: projection workload

Measures a narrow transformation without shuffle.

In [8]:
projected = fact_sales.select(
    "sale_id",
    "customer_id",
    (F.col("quantity") * F.col("amount")).alias("gross_amount")
)

projected.explain("formatted")
record(stage_metrics("03_projection_count", lambda: projected.count()))
results()

== Physical Plan ==
* Project (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [3]: [id#0L AS sale_id#1L, (id#0L % 200000) AS customer_id#2L, (cast(cast(((id#0L % 5) + 1) as int) as double) * cast(((id#0L % 500) + 10) as double)) AS gross_amount#67]
Input [1]: [id#0L]



--- 03_projection_count ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 2
numTasks => 3
elapsedTime => 71 (71 ms)
stageDuration => 55 (55 ms)
executorRunTime => 11 (11 ms)
executorCpuTime => 9 (9 ms)
executorDeserializeTime => 34 (34 ms)
executorDeserializeCpuTime => 14 (14 ms)
resultSerializationTime => 0 (0 ms)
jvmGCTime => 0 (0 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
resultSize => 3850 (3.8 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 67174368
recordsRead => 2000

,case,seconds
3,03_projection_count,0.097
1,01_count_baseline,0.115
0,01_count_baseline,0.138
2,02_filter_count,0.181


dsWritten => 2

Average number of active tasks => 0.2

Stages and their duration:
Stage 18 duration => 27 (27 ms)
Stage 20 duration => 28 (28 ms)


## 07 — Example 04: aggregation with shuffle

Measures groupBy aggregation and shuffle cost.

In [9]:
by_store = (
    fact_sales
    .groupBy("store_id")
    .agg(
        F.count("*").alias("sales"),
        F.round(F.sum("amount"), 2).alias("revenue")
    )
)

by_store.explain("formatted")
record(stage_metrics("04_groupby_shuffle", lambda: by_store.count()))
results()

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Project
Output [2]: [(id#0L % 1000) AS store_id#4L, cast(((id#0L % 500) + 10) as double) AS amount#6]
Input [1]: [id#0L]

(3) HashAggregate
Input [2]: [store_id#4L, amount#6]
Keys [1]: [store_id#4L]
Functions [2]: [partial_count(1), partial_sum(amount#6)]
Aggregate Attributes [2]: [count#85L, sum#86]
Results [3]: [store_id#4L, count#87L, sum#88]

(4) Exchange
Input [3]: [store_id#4L, count#87L, sum#88]
Arguments: hashpartitioning(store_id#4L, 64), ENSURE_REQUIREMENTS, [plan_id=272]

(5) HashAggregate
Input [3]: [store_id#4L, count#87L, sum#88]
Keys [1]: [store_id#4L]
Functions [2]: [count(1), sum(amount#6)]
Aggregate Attributes [2]: [count(1)#83L, sum(amount#6)#84]
Results [3]: [store_id#4L, count(1)#83L AS sales#75L, round(sum(amount#6)#84,

,case,seconds
3,03_projection_count,0.097
1,01_count_baseline,0.115
0,01_count_baseline,0.138
2,02_filter_count,0.181
4,04_groupby_shuffle,1.128


## 08 — Example 05: shuffle partition tuning

Runs the same aggregation with different shuffle partition counts.

In [10]:
for partitions in [8, 32, 64, 128]:
    spark.conf.set("spark.sql.shuffle.partitions", str(partitions))
    df = fact_sales.groupBy("store_id").agg(F.sum("amount").alias("revenue"))
    row = stage_metrics(f"05_shuffle_partitions_{partitions}", lambda df=df: df.count())
    row["shuffle_partitions"] = partitions
    record(row)

spark.conf.set("spark.sql.shuffle.partitions", "64")
results()


--- 05_shuffle_partitions_8 ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 3
numTasks => 4
elapsedTime => 232 (0.2 s)
stageDuration => 190 (0.2 s)
executorRunTime => 141 (0.1 s)
executorCpuTime => 124 (0.1 s)
executorDeserializeTime => 68 (68 ms)
executorDeserializeCpuTime => 32 (32 ms)
resultSerializationTime => 4 (4 ms)
jvmGCTime => 0 (0 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
resultSize => 5092 (5.0 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 101547984
recordsRead => 2000000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2001
shuffleTotalBlocksFetched => 3
shuffleLocalBlocksFetched => 3
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 13437 (13.1 KB)
shuffleLocalBytesRead => 13437 (13.1 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleB

,case,seconds,shuffle_partitions
3,03_projection_count,0.097,NaN
1,01_count_baseline,0.115,NaN
0,01_count_baseline,0.138,NaN
2,02_filter_count,0.181,NaN
7,05_shuffle_partitions_64,0.303,64.0
5,05_shuffle_partitions_8,0.308,8.0
8,05_shuffle_partitions_128,0.311,128.0
6,05_shuffle_partitions_32,0.316,32.0
4,04_groupby_shuffle,1.128,NaN


## 09 — Example 06: shuffle join

Measures SortMergeJoin-style behavior with broadcast disabled.

In [11]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

shuffle_join = fact_sales.join(customers, "customer_id")

shuffle_join.explain("formatted")
record(stage_metrics("06_shuffle_join", lambda: shuffle_join.count()))
results()

== Physical Plan ==
* Project (11)
+- * SortMergeJoin Inner (10)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Project (3)
   :        +- * Filter (2)
   :           +- * Range (1)
   +- * Sort (9)
      +- Exchange (8)
         +- * Project (7)
            +- * Range (6)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 200000))

(3) Project [codegen id : 1]
Output [6]: [id#0L AS sale_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 50000) AS product_id#3L, (id#0L % 1000) AS store_id#4L, cast(((id#0L % 5) + 1) as int) AS quantity#5, cast(((id#0L % 500) + 10) as double) AS amount#6]
Input [1]: [id#0L]

(4) Exchange
Input [6]: [sale_id#1L, customer_id#2L, product_id#3L, store_id#4L, quantity#5, amount#6]
Arguments: hashpartitioning(customer_id#2L, 64), ENSURE_REQUIREMENTS, [plan_id=744]

(5) Sort [codegen id : 2]
Input [6]: [sale_id#1L, customer_i

[Stage 53:==========================================>             (49 + 3) / 64]


--- 06_shuffle_join ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 4
numTasks => 69
elapsedTime => 2284 (2 s)
stageDuration => 2469 (2 s)
executorRunTime => 3469 (3 s)
executorCpuTime => 3131 (3 s)
executorDeserializeTime => 443 (0.4 s)
executorDeserializeCpuTime => 373 (0.4 s)
resultSerializationTime => 17 (17 ms)
jvmGCTime => 142 (0.1 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 44 (44 ms)
resultSize => 395871 (386.6 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 6692008896
recordsRead => 2200000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2200064
shuffleTotalBlocksFetched => 320
shuffleLocalBlocksFetched => 320
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 11904901 (11.4 MB)
shuffleLocalBytesRead => 11904901 (11.4 MB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0

,case,seconds,shuffle_partitions
3,03_projection_count,0.097,NaN
1,01_count_baseline,0.115,NaN
0,01_count_baseline,0.138,NaN
2,02_filter_count,0.181,NaN
7,05_shuffle_partitions_64,0.303,64.0
5,05_shuffle_partitions_8,0.308,8.0
8,05_shuffle_partitions_128,0.311,128.0
6,05_shuffle_partitions_32,0.316,32.0
4,04_groupby_shuffle,1.128,NaN
9,06_shuffle_join,2.542,NaN


## 10 — Example 07: broadcast join

Forces BroadcastHashJoin and compares metrics with the shuffle join.

In [12]:
broadcast_join = fact_sales.join(F.broadcast(customers), "customer_id")

broadcast_join.explain("formatted")
record(stage_metrics("07_broadcast_join", lambda: broadcast_join.count()))
results()

== Physical Plan ==
* Project (8)
+- * BroadcastHashJoin Inner BuildRight (7)
   :- * Project (3)
   :  +- * Filter (2)
   :     +- * Range (1)
   +- BroadcastExchange (6)
      +- * Project (5)
         +- * Range (4)


(1) Range [codegen id : 2]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 2]
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 200000))

(3) Project [codegen id : 2]
Output [6]: [id#0L AS sale_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 50000) AS product_id#3L, (id#0L % 1000) AS store_id#4L, cast(((id#0L % 5) + 1) as int) AS quantity#5, cast(((id#0L % 500) + 10) as double) AS amount#6]
Input [1]: [id#0L]

(4) Range [codegen id : 1]
Output [1]: [id#7L]
Arguments: Range (0, 200000, step=1, splits=Some(2))

(5) Project [codegen id : 1]
Output [3]: [id#7L AS customer_id#8L, CASE WHEN ((id#7L % 5) = 0) THEN SK WHEN ((id#7L % 5) = 1) THEN CZ WHEN ((id#7L % 5) = 2) THEN DE WHEN ((id#7L % 5) = 3) THEN AT ELSE PL END 

,case,seconds,shuffle_partitions
3,03_projection_count,0.097,NaN
1,01_count_baseline,0.115,NaN
0,01_count_baseline,0.138,NaN
2,02_filter_count,0.181,NaN
7,05_shuffle_partitions_64,0.303,64.0
5,05_shuffle_partitions_8,0.308,8.0
8,05_shuffle_partitions_128,0.311,128.0
6,05_shuffle_partitions_32,0.316,32.0
10,07_broadcast_join,0.516,NaN
4,04_groupby_shuffle,1.128,NaN


## 11 — Example 08: multi-join query

Measures a fact table joined with two dimensions.

In [13]:
multi_join = (
    fact_sales
    .join(customers, "customer_id")
    .join(products, "product_id")
    .groupBy("country", "segment", "category")
    .agg(F.count("*").alias("sales"), F.round(F.sum("amount"), 2).alias("revenue"))
)

multi_join.explain("formatted")
record(stage_metrics("08_multi_join_aggregation", lambda: multi_join.count()))
results()

== Physical Plan ==
* HashAggregate (22)
+- Exchange (21)
   +- * HashAggregate (20)
      +- * Project (19)
         +- * SortMergeJoin Inner (18)
            :- * Sort (13)
            :  +- Exchange (12)
            :     +- * Project (11)
            :        +- * SortMergeJoin Inner (10)
            :           :- * Sort (5)
            :           :  +- Exchange (4)
            :           :     +- * Project (3)
            :           :        +- * Filter (2)
            :           :           +- * Range (1)
            :           +- * Sort (9)
            :              +- Exchange (8)
            :                 +- * Project (7)
            :                    +- * Range (6)
            +- * Sort (17)
               +- Exchange (16)
                  +- * Project (15)
                     +- * Range (14)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : (isnotn

[Stage 62:==================================================>     (58 + 2) / 64]


--- 08_multi_join_aggregation ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 7
numTasks => 199
elapsedTime => 4426 (4 s)
stageDuration => 4688 (5 s)
executorRunTime => 6807 (7 s)
executorCpuTime => 6234 (6 s)
executorDeserializeTime => 699 (0.7 s)
executorDeserializeCpuTime => 694 (0.7 s)
resultSerializationTime => 38 (38 ms)
jvmGCTime => 364 (0.4 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 139 (0.1 s)
resultSize => 678065 (662.2 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 15494800288
recordsRead => 2250000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 4251344
shuffleTotalBlocksFetched => 5760
shuffleLocalBlocksFetched => 5760
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 28451176 (27.1 MB)
shuffleLocalBytesRead => 28451176 (27.1 MB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesRea

,case,seconds,shuffle_partitions
3,03_projection_count,0.097,NaN
1,01_count_baseline,0.115,NaN
0,01_count_baseline,0.138,NaN
2,02_filter_count,0.181,NaN
7,05_shuffle_partitions_64,0.303,64.0
5,05_shuffle_partitions_8,0.308,8.0
8,05_shuffle_partitions_128,0.311,128.0
6,05_shuffle_partitions_32,0.316,32.0
10,07_broadcast_join,0.516,NaN
4,04_groupby_shuffle,1.128,NaN


## 12 — Example 09: AQE on vs off

Compares the same query with Adaptive Query Execution disabled and enabled.

In [14]:
aqe_rows = []

for enabled in [False, True]:
    spark.conf.set("spark.sql.adaptive.enabled", "true" if enabled else "false")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
    df = fact_sales.join(customers, "customer_id").groupBy("country").agg(F.sum("amount").alias("revenue"))
    row = stage_metrics(f"09_aqe_{enabled}", lambda df=df: df.count())
    row["aqe_enabled"] = enabled
    record(row)
    aqe_rows.append(row)

pd.DataFrame(aqe_rows).drop(columns=["result"], errors="ignore")


--- 09_aqe_False ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 5
numTasks => 133
elapsedTime => 1448 (1 s)
stageDuration => 1784 (2 s)
executorRunTime => 2051 (2 s)
executorCpuTime => 1959 (2 s)
executorDeserializeTime => 232 (0.2 s)
executorDeserializeCpuTime => 250 (0.3 s)
resultSerializationTime => 23 (23 ms)
jvmGCTime => 96 (96 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 28 (28 ms)
resultSize => 502660 (490.9 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 8875143104
recordsRead => 2200000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2200384
shuffleTotalBlocksFetched => 640
shuffleLocalBlocksFetched => 640
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 12397004 (11.8 MB)
shuffleLocalBytesRead => 12397004 (11.8 MB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 By

,case,seconds,aqe_enabled
0,09_aqe_False,1.570,False
1,09_aqe_True,0.913,True


## 13 — Example 10: cache effect

Measures an aggregation before and after caching the source DataFrame.

In [15]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

source = fact_sales.repartition(64, "customer_id")

uncached = source.groupBy("store_id").agg(F.sum("amount").alias("revenue"))
record(stage_metrics("10_uncached_groupby", lambda: uncached.count()))

source.cache()
source.count()

cached = source.groupBy("store_id").agg(F.sum("amount").alias("revenue"))
record(stage_metrics("10_cached_groupby", lambda: cached.count()))

source.unpersist()

results()


--- 10_uncached_groupby ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 4
numTasks => 68
elapsedTime => 1041 (1 s)
stageDuration => 971 (1.0 s)
executorRunTime => 1420 (1 s)
executorCpuTime => 1352 (1 s)
executorDeserializeTime => 94 (94 ms)
executorDeserializeCpuTime => 96 (96 ms)
resultSerializationTime => 21 (21 ms)
jvmGCTime => 71 (71 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 51 (51 ms)
resultSize => 290863 (284.0 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 2300869584
recordsRead => 2000000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2061332
shuffleTotalBlocksFetched => 193
shuffleLocalBlocksFetched => 193
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 20988011 (20.0 MB)
shuffleLocalBytesRead => 20988011 (20.0 MB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 


--- 10_cached_groupby ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 3
numTasks => 66
elapsedTime => 519 (0.5 s)
stageDuration => 494 (0.5 s)
executorRunTime => 594 (0.6 s)
executorCpuTime => 515 (0.5 s)
executorDeserializeTime => 79 (79 ms)
executorDeserializeCpuTime => 85 (85 ms)
resultSerializationTime => 15 (15 ms)
jvmGCTime => 90 (90 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 15 (15 ms)
resultSize => 249901 (244.0 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 2200206320
recordsRead => 256
bytesRead => 35531696 (33.9 MB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 61332
shuffleTotalBlocksFetched => 65
shuffleLocalBlocksFetched => 65
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 655817 (640.4 KB)
shuffleLocalBytesRead => 655817 (640.4 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0

,case,seconds,shuffle_partitions,aqe_enabled
3,03_projection_count,0.097,NaN,NaN
1,01_count_baseline,0.115,NaN,NaN
0,01_count_baseline,0.138,NaN,NaN
2,02_filter_count,0.181,NaN,NaN
7,05_shuffle_partitions_64,0.303,64.0,NaN
5,05_shuffle_partitions_8,0.308,8.0,NaN
8,05_shuffle_partitions_128,0.311,128.0,NaN
6,05_shuffle_partitions_32,0.316,32.0,NaN
10,07_broadcast_join,0.516,NaN,NaN
15,10_cached_groupby,0.561,NaN,NaN


## 14 — Example 11: repartition vs coalesce

Measures repartitioning and coalescing effects.

In [16]:
repartitioned = fact_sales.repartition(128, "customer_id")
coalesced = fact_sales.coalesce(8)

record(stage_metrics("11_repartition_128", lambda: repartitioned.count()))
record(stage_metrics("11_coalesce_8", lambda: coalesced.count()))

results()


--- 11_repartition_128 ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 3
numTasks => 131
elapsedTime => 758 (0.8 s)
stageDuration => 732 (0.7 s)
executorRunTime => 980 (1.0 s)
executorCpuTime => 981 (1.0 s)
executorDeserializeTime => 24 (24 ms)
executorDeserializeCpuTime => 95 (95 ms)
resultSerializationTime => 5 (5 ms)
jvmGCTime => 39 (39 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 21 (21 ms)
resultSize => 509444 (497.5 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 4399822816
recordsRead => 2000000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2000128
shuffleTotalBlocksFetched => 384
shuffleLocalBlocksFetched => 384
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 10979637 (10.5 MB)
shuffleLocalBytesRead => 10979637 (10.5 MB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0

,case,seconds,shuffle_partitions,aqe_enabled
3,03_projection_count,0.097,NaN,NaN
1,01_count_baseline,0.115,NaN,NaN
17,11_coalesce_8,0.118,NaN,NaN
0,01_count_baseline,0.138,NaN,NaN
2,02_filter_count,0.181,NaN,NaN
7,05_shuffle_partitions_64,0.303,64.0,NaN
5,05_shuffle_partitions_8,0.308,8.0,NaN
8,05_shuffle_partitions_128,0.311,128.0,NaN
6,05_shuffle_partitions_32,0.316,32.0,NaN
10,07_broadcast_join,0.516,NaN,NaN


## 15 — Example 12: task metrics

Uses TaskMetrics instead of StageMetrics for a shuffle aggregation.

In [17]:
task_df = fact_sales.groupBy("product_id").agg(F.sum("amount").alias("revenue"))

record(task_metrics("12_task_metrics_groupby", lambda: task_df.count()))
results()


--- 12_task_metrics_groupby ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark task metrics:
numTasks => 4
successful tasks => 4
speculative tasks => 0
taskDuration => 270 (0.3 s)
schedulerDelayTime => 19 (19 ms)
executorRunTime => 229 (0.2 s)
executorCpuTime => 209 (0.2 s)
executorDeserializeTime => 22 (22 ms)
executorDeserializeCpuTime => 9 (9 ms)
resultSerializationTime => 0 (0 ms)
jvmGCTime => 0 (0 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 1 (1 ms)
gettingResultTime => 0 (0 ms)
resultSize => 5049 (4.9 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 103579600
recordsRead => 2000000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 100001
shuffleTotalBlocksFetched => 3
shuffleLocalBlocksFetched => 3
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 637131 (622.2 KB)
shuffleLocalBytesRead => 637131 (622.2 KB)
shuffleRemoteBytesRead

,case,seconds,shuffle_partitions,aqe_enabled
3,03_projection_count,0.097,NaN,NaN
1,01_count_baseline,0.115,NaN,NaN
17,11_coalesce_8,0.118,NaN,NaN
0,01_count_baseline,0.138,NaN,NaN
2,02_filter_count,0.181,NaN,NaN
18,12_task_metrics_groupby,0.243,NaN,NaN
7,05_shuffle_partitions_64,0.303,64.0,NaN
5,05_shuffle_partitions_8,0.308,8.0,NaN
8,05_shuffle_partitions_128,0.311,128.0,NaN
6,05_shuffle_partitions_32,0.316,32.0,NaN


## 16 — Example 13: skew simulation

Creates a hot key and measures the skewed join.

In [18]:
normal_events = (
    spark.range(0, 500_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("customer_id", ((F.col("event_id") % (CUSTOMER_ROWS - 1)) + 1).cast("long"))
)

hot_events = (
    spark.range(0, 1_500_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("customer_id", F.lit(1).cast("long"))
)

skewed_events = normal_events.unionByName(hot_events)

skewed_events.groupBy("customer_id").count().orderBy(F.desc("count")).show(5)

spark.conf.set("spark.sql.adaptive.enabled", "false")
skewed_join = skewed_events.join(customers, "customer_id")

skewed_join.explain("formatted")
record(stage_metrics("13_skewed_join_no_aqe", lambda: skewed_join.count()))

results()

+-----------+-------+
|customer_id|  count|
+-----------+-------+
|          1|1500003|
|         41|      3|
|         93|      3|
|         52|      3|
|        171|      3|
+-----------+-------+
only showing top 5 rows
== Physical Plan ==
* Project (14)
+- * SortMergeJoin Inner (13)
   :- * Sort (8)
   :  +- Exchange (7)
   :     +- Union (6)
   :        :- * Project (3)
   :        :  +- * Filter (2)
   :        :     +- * Range (1)
   :        +- * Project (5)
   :           +- * Range (4)
   +- * Sort (12)
      +- Exchange (11)
         +- * Project (10)
            +- * Range (9)


(1) Range [codegen id : 1]
Output [1]: [id#637L]
Arguments: Range (0, 500000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#637L]
Condition : isnotnull(((id#637L % 199999) + 1))

(3) Project [codegen id : 1]
Output [2]: [id#637L AS event_id#638L, ((id#637L % 199999) + 1) AS customer_id#639L]
Input [1]: [id#637L]

(4) Range [codegen id : 2]
Output [1]: [id#640L]
Arguments: Range 

,case,seconds,shuffle_partitions,aqe_enabled
3,03_projection_count,0.097,NaN,NaN
1,01_count_baseline,0.115,NaN,NaN
17,11_coalesce_8,0.118,NaN,NaN
0,01_count_baseline,0.138,NaN,NaN
2,02_filter_count,0.181,NaN,NaN
18,12_task_metrics_groupby,0.243,NaN,NaN
7,05_shuffle_partitions_64,0.303,64.0,NaN
5,05_shuffle_partitions_8,0.308,8.0,NaN
8,05_shuffle_partitions_128,0.311,128.0,NaN
6,05_shuffle_partitions_32,0.316,32.0,NaN


## 17 — Example 14: AQE skew handling

Enables AQE skew join handling and measures the same skewed join.

In [19]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1024")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

skewed_join_aqe = skewed_events.join(customers, "customer_id")

record(stage_metrics("14_skewed_join_aqe", lambda: skewed_join_aqe.count()))
skewed_join_aqe.explain("formatted")

results()


--- 14_skewed_join_aqe ---

Scheduling mode = FIFO
Spark Context default degree of parallelism = 2

Aggregated Spark stage metrics:
numStages => 4
numTasks => 9
elapsedTime => 700 (0.7 s)
stageDuration => 713 (0.7 s)
executorRunTime => 938 (0.9 s)
executorCpuTime => 925 (0.9 s)
executorDeserializeTime => 48 (48 ms)
executorDeserializeCpuTime => 16 (16 ms)
resultSerializationTime => 0 (0 ms)
jvmGCTime => 10 (10 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 7 (7 ms)
resultSize => 12524 (12.2 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 574684976
recordsRead => 2200000
bytesRead => 0 (0 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2200002
shuffleTotalBlocksFetched => 12
shuffleLocalBlocksFetched => 12
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 4029442 (3.8 MB)
shuffleLocalBytesRead => 4029442 (3.8 MB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
sh

,case,seconds,shuffle_partitions,aqe_enabled
3,03_projection_count,0.097,NaN,NaN
1,01_count_baseline,0.115,NaN,NaN
17,11_coalesce_8,0.118,NaN,NaN
0,01_count_baseline,0.138,NaN,NaN
2,02_filter_count,0.181,NaN,NaN
18,12_task_metrics_groupby,0.243,NaN,NaN
7,05_shuffle_partitions_64,0.303,64.0,NaN
5,05_shuffle_partitions_8,0.308,8.0,NaN
8,05_shuffle_partitions_128,0.311,128.0,NaN
6,05_shuffle_partitions_32,0.316,32.0,NaN


## 18 — Example 15: manual salting

Measures manual salting for one dominant hot key.

In [20]:
SALT_BUCKETS = 8

salted_events = skewed_events.withColumn(
    "salt",
    F.when(F.col("customer_id") == 1, (F.rand(seed=42) * SALT_BUCKETS).cast("int")).otherwise(F.lit(0))
)

salt_array = F.when(
    F.col("customer_id") == 1,
    F.array(*[F.lit(i) for i in range(SALT_BUCKETS)])
).otherwise(F.array(F.lit(0)))

salted_customers = customers.withColumn("salt", F.explode(salt_array))

spark.conf.set("spark.sql.adaptive.enabled", "false")

salted_join = salted_events.join(salted_customers, ["customer_id", "salt"])

salted_join.explain("formatted")
record(stage_metrics("15_manual_salting", lambda: salted_join.count()))

results()

== Physical Plan ==
* Project (16)
+- * SortMergeJoin Inner (15)
   :- * Sort (9)
   :  +- Exchange (8)
   :     +- * Filter (7)
   :        +- * Project (6)
   :           +- Union (5)
   :              :- * Project (2)
   :              :  +- * Range (1)
   :              +- * Project (4)
   :                 +- * Range (3)
   +- * Sort (14)
      +- Exchange (13)
         +- * Generate (12)
            +- * Project (11)
               +- * Range (10)


(1) Range [codegen id : 1]
Output [1]: [id#637L]
Arguments: Range (0, 500000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [2]: [id#637L AS event_id#638L, ((id#637L % 199999) + 1) AS customer_id#639L]
Input [1]: [id#637L]

(3) Range [codegen id : 2]
Output [1]: [id#640L]
Arguments: Range (0, 1500000, step=1, splits=Some(2))

(4) Project [codegen id : 2]
Output [2]: [id#640L AS event_id#641L, 1 AS customer_id#642L]
Input [1]: [id#640L]

(5) Union

(6) Project [codegen id : 3]
Output [3]: [event_id#638L, customer_id#639L

,case,seconds,shuffle_partitions,aqe_enabled
3,03_projection_count,0.097,NaN,NaN
1,01_count_baseline,0.115,NaN,NaN
17,11_coalesce_8,0.118,NaN,NaN
0,01_count_baseline,0.138,NaN,NaN
2,02_filter_count,0.181,NaN,NaN
18,12_task_metrics_groupby,0.243,NaN,NaN
7,05_shuffle_partitions_64,0.303,64.0,NaN
5,05_shuffle_partitions_8,0.308,8.0,NaN
8,05_shuffle_partitions_128,0.311,128.0,NaN
6,05_shuffle_partitions_32,0.316,32.0,NaN


## 19 — Final benchmark results

Sorted runtime table for all recorded benchmark cases.

In [21]:
results()

,case,seconds,shuffle_partitions,aqe_enabled
3,03_projection_count,0.097,NaN,NaN
1,01_count_baseline,0.115,NaN,NaN
17,11_coalesce_8,0.118,NaN,NaN
0,01_count_baseline,0.138,NaN,NaN
2,02_filter_count,0.181,NaN,NaN
18,12_task_metrics_groupby,0.243,NaN,NaN
7,05_shuffle_partitions_64,0.303,64.0,NaN
5,05_shuffle_partitions_8,0.308,8.0,NaN
8,05_shuffle_partitions_128,0.311,128.0,NaN
6,05_shuffle_partitions_32,0.316,32.0,NaN
